# Batches with Airflow

Why books are processed in hourly batches, and the idempotent steps that make re-running cheap.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

## The pipeline steps and their keys (offline)
Each step is skipped while the hash of its inputs is unchanged.

In [ ]:
import tempfile, pathlib
from qdrant_client import QdrantClient
from marginalia.config import Settings
from marginalia.ingest.pipeline import Pipeline, STEPS
from marginalia.library import Library
from tests.conftest import FakeDense, FakeSparse
from tests.test_pipeline import FakeSummarizer
from tests.pdfs import make_book, chapter
library = Library(pathlib.Path(tempfile.mkdtemp()) / 'library')
pipe = Pipeline(library, Settings(collection='batch-notebook', covers_online=False), client=QdrantClient(':memory:'),
                dense=FakeDense(), sparse=FakeSparse(), summarizer=FakeSummarizer(), log=print)
book = library.add(make_book([chapter('CHAPTER I', 'the river'), chapter('CHAPTER II', 'the axe')], title='Batch Demo'))
print('steps:', STEPS)
print('ran:', pipe.run(book.book_id))
print('again:', pipe.run(book.book_id), '(up to date)')
print({k: v['key'] for k, v in library.get(book.book_id).steps.items()})

## The DAG

In [ ]:
print(open('dags/marginalia_ingest.py', encoding='utf-8').read()[:2500])

## What the next batch would do, and the runs (needs the stack)

In [ ]:
import httpx
from marginalia.config import Settings
s = Settings()
api = httpx.Client(base_url='http://127.0.0.1:8010/api', timeout=30)
token = api.post('/auth/login', json={'email': s.admin_email, 'password': s.admin_password}).json()['access_token']
api.headers['Authorization'] = f'Bearer {token}'
print(api.get('/admin/pipeline/waiting').json())
for run in api.get('/admin/pipeline', params={'limit': 5}).json()['runs']:
    print(run['run_id'], run['state'], run.get('duration'))